In [77]:
import re
import numpy as np
import pandas as pd

raw = pd.read_csv('week01_customers.csv', dtype=str)

NULL_TOKENS = ["NaN", "unknown", "-"]

CITY_ALIASES = {
    "nyc": "New York",
    "new york city": "New York",
}

DATE_FORMATS = [
    "%Y-%m-%d",
    "%m/%d/%Y",
    "%b %d %Y",
]

EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[A-Za-z]{2,}$")

# Part 1

### Task 1
Every column is an object because they are either all strings or mixed datatypes.
Age should be numeric, but some people have spelled out their age, making it a string
Price should be numeric, but sometimes there is a $ in front of price

### Task 2
df.isna().sum() does not give the full picture, because some values are not usable, yet do not show up through the command

### Task 3
C-1005 and C-1013 are repeated. They are completely the same for every value

### Task 4
Naive counts shown in the last cell
Real counts:
Boston      12
Chicago     11
Seattle     11
Atlanta     10
New York     13

In [78]:
print(raw.shape)
print("-")
print(raw.dtypes)
print("-")
print(raw.isna().sum())

(57, 8)
-
customer_id    object
name           object
age            object
city           object
signup_date    object
price          object
email          object
membership     object
dtype: object
-
customer_id    0
name           0
age            5
city           0
signup_date    0
price          4
email          4
membership     0
dtype: int64


In [79]:
print(raw)

   customer_id             name        age      city  signup_date      price  \
0       C-1001        Ann Smith         25  New York   2026-01-15  $1,299.00   
1       C-1002      Brian Ochoa     thirty    Boston   2026-01-17      45.50   
2       C-1003      Carla Reyes         41   Chicago   2026-01-19     230.00   
3       C-1004        Devon Lee        NaN    Boston   2026-01-21     $89.99   
4       C-1005       Elena Ford         34  new york   2026-01-22   1,050.00   
5       C-1006     Frank Nguyen         28   Chicago   01/23/2026      62.75   
6       C-1007    Grace Adeyemi        NaN   Seattle   2026-01-25     310.20   
7       C-1008      Hector Diaz         52   Boston    2026-01-26  USD 75.00   
8       C-1009     Imani Brooks  forty-two   Atlanta   2026-01-28  $2,480.00   
9       C-1010      Jonas Weber         39   Seattle   2026-01-29      18.00   
10      C-1011    Keisha Palmer         31   Atlanta   2026-02-01        NaN   
11      C-1012   Luis Marchetti         

In [80]:
print(len(raw))
print(raw['customer_id'].nunique())
print(raw['customer_id'].duplicated(keep=False))

57
55
0     False
1     False
2     False
3     False
4      True
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12     True
13    False
14    False
15    False
16    False
17    False
18    False
19    False
20    False
21    False
22    False
23    False
24    False
25    False
26    False
27    False
28    False
29    False
30    False
31    False
32    False
33    False
34    False
35    False
36    False
37    False
38    False
39    False
40    False
41    False
42    False
43    False
44    False
45    False
46    False
47    False
48    False
49    False
50    False
51    False
52    False
53    False
54    False
55     True
56     True
Name: customer_id, dtype: bool


In [81]:
print(raw["city"].value_counts())

Boston      11
Chicago     11
Seattle     11
Atlanta     10
new york     6
NYC          4
NEW YORK     2
New York     1
 Boston      1
Name: city, dtype: int64


# Part 2

### Task 5
I included NaN, unknown, and -, because they are the tokens that represent an unknown value in this dataset

### Task 6
a) None are UNUSABLE, but some are null / NaN (5)
b) C-1041
c) There will be many different types of unusable numbers, you must take into account a wide range

### Task 7
a) Easier to understand / consolidate
b) The naive call discarded null values
c) Between 16 and 100 because a person cannot physically buy things outside of this range
d) The mean before my range was much larger because of the 999 outlier

### Task 8
Final counts same as before, .str.title is insufficient because it does not take into account apostrophes and other particularities.

### Task 9
There is YYYY-MM-DD, MM/DD/YYYY, and MM DD YYYY.
01/23/2026 is unambiguous because the beginning month can only be a month not a day, since 23 cannot be a month

### Task 10
4 addresses are missing (C-1006, C-1017, C-1026, C-1043). Of the 53 present, all 53 pass the regex.
Gabriel Garcia Marquez breaks because it is 3 parts
Mao Zedong breaks because family name comes first

In [82]:
def to_null(value):
    
    if value in NULL_TOKENS:
        return np.nan
    if isinstance(value, str):
        value.strip()
        
    return value

def clean_price(value):
    if isinstance(value, str):
        value = value.strip()
        value = value.strip("$")
        value = value.strip("USD")
        value = value.replace(",", "")
        value = float(value)
    
    return value

def clean_age(value):
    value = float(value)
    return value

def validate_age(value):
    value = to_null(value)
    if value < 16 or value > 100:
        value = np.nan
    
    return value

# Claude Used
def clean_city(value):
    value = to_null(value)
    if not isinstance(value, str):
        return value

    value = value.strip()
    value = re.sub(r"\s+", " ", value)

    key = value.lower()
    if key in CITY_ALIASES:
        return CITY_ALIASES[key]

    return value.title()

# Claude Used
def clean_date(value):
    value = to_null(value)
    if not isinstance(value, str):
        return pd.NaT

    value = value.strip()
    value = re.sub(r"\s+", " ", value)

    for fmt in DATE_FORMATS:
        try:
            return pd.to_datetime(value, format=fmt)
        except ValueError:
            pass

    return pd.NaT

# Claude Used
def valid_email(value):
    value = to_null(value)
    if not isinstance(value, str):
        return False

    return EMAIL_RE.match(value.strip()) is not None


In [83]:
# email validity
raw["email_valid"] = raw["email"].map(valid_email)

present = raw["email"].notna()
print("missing emails:", int((~present).sum()))
print("present emails:", int(present.sum()))
print("present and passing:", int((present & raw["email_valid"]).sum()))
print("present and failing:", int((present & ~raw["email_valid"]).sum()))

# split name into first / last
n_parts = raw["name"].str.strip().str.split(r"\s+", regex=True).str.len()
print("every row splits into exactly two parts:", bool(n_parts.eq(2).all()))
print(n_parts.value_counts())

parts = raw["name"].str.strip().str.split(r"\s+", expand=True, regex=True)
raw["first_name"] = parts[0]
raw["last_name"] = parts[1]

print(raw[["name", "first_name", "last_name", "email", "email_valid"]].head())

missing emails: 4
present emails: 53
present and passing: 53
present and failing: 0
every row splits into exactly two parts: True
2    57
Name: name, dtype: int64
          name first_name last_name                  email  email_valid
0    Ann Smith        Ann     Smith  ann.smith@example.com         True
1  Brian Ochoa      Brian     Ochoa    b.ochoa@example.com         True
2  Carla Reyes      Carla     Reyes    carla.r@example.com         True
3    Devon Lee      Devon       Lee  devon.lee@example.com         True
4   Elena Ford      Elena      Ford     e.ford@example.com         True


## Part 3
